In [ ]:
# =============================================================================
# Parkinson's Disease - Cross-Sectional ML Analysis Pipeline
#
# Description:
#   Complete cross-sectional analysis for PD classification using
#   alpha-synuclein biomarkers (THT + DLS) and demographics.
#
# Tasks:
#   Task 1: HC vs PD        (16 features, no disease duration)
#   Task 2: PD-NC vs PD-CI  (17 features, with disease duration)
#   Task 3: 4-class         (HC / PD-NC / PD-MCI / PD-D)
#
# Phases:
#   Phase 1: Univariate     (single feature, 7 models)
#   Phase 2: Top-K          (RF importance-ranked, multiple K values)
#   Phase 3: Predefined     (feature subsets: THT-only, DLS-only, etc.)
#
# Validation:
#   CV:   RepeatedStratifiedKFold (5-fold x 10 repeats)
#   LOCO: Leave-One-Cohort-Out (JHU <-> UW)
#
# Input:
#   - 02092026_cross_sectional.csv
#
# Output:
#   - demographics_statistics.xlsx
#   - biomarker_validation.xlsx
#   - biomarker_validation_figures.pdf
#   - /content/final_corrected_results/
#       {Task}_Phase1_Results.xlsx
#       {Task}_Phase2_Results.xlsx
#       {Task}_Phase2_FeatureImportance.xlsx
#       {Task}_Phase2_ConfusionMatrices.xlsx
#       {Task}_Phase2_ClassificationReports.xlsx
#       {Task}_Phase3_Results.xlsx
#       {Task}_Phase3_ConfusionMatrices.xlsx
#       {Task}_Phase3_ClassificationReports.xlsx
#       LeaveOneCohortOut_Results.xlsx
#
# =============================================================================

# -----------------------------------------------------------------------------
# Install dependencies (run once)
# -----------------------------------------------------------------------------
# !pip install xgboost catboost openpyxl

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal
from itertools import combinations
from pathlib import Path
from copy import deepcopy

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier)
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                              precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

print("=" * 70)
print("PD CROSS-SECTIONAL ML ANALYSIS PIPELINE")
print("=" * 70)

# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_PATH  = '/content/02092026_cross_sectional.csv'
OUTPUT_DIR = '/content/final_corrected_results'

OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Feature definitions
BIOMARKERS = {
    'THT': ['tht-mfi', 'tht-tlag', 'tht-T20', 'tht-t50', 'forming rate'],
    'DLS': ['dls-peak-number', 'dls-peak-1-intensity', 'dls-peak-2-intensity',
            'dls-peak-1-size', 'dls-peak-2-size', 'dls-peak-1-pd', 'dls-peak-2-pd'],
    'Neurotoxicity': ['neurotoxicity']
}

ALL_BIOMARKERS         = BIOMARKERS['THT'] + BIOMARKERS['DLS'] + BIOMARKERS['Neurotoxicity']
DEMOGRAPHICS_NO_DURATION = ['age', 'sex', 'education']
DEMOGRAPHICS_ALL       = ['age', 'sex', 'education', 'disease duration']

FEATURES_TASK1  = ALL_BIOMARKERS + DEMOGRAPHICS_NO_DURATION  # 16 features
FEATURES_TASK23 = ALL_BIOMARKERS + DEMOGRAPHICS_ALL          # 17 features

print(f"Task 1 features:   {len(FEATURES_TASK1)}  (no disease duration)")
print(f"Task 2,3 features: {len(FEATURES_TASK23)} (with disease duration)")

# =============================================================================
# PART 1: STATISTICAL VALIDATION
#         Demographics + Biomarker group comparisons
# =============================================================================

print("\n" + "=" * 70)
print("PART 1: STATISTICAL VALIDATION")
print("=" * 70)

df = pd.read_csv(DATA_PATH)
print(f"\nData loaded: {df.shape[0]} samples, {df.shape[1]} columns")

labels_sorted = sorted(df['label'].unique())
print(f"Labels: {labels_sorted}")

# -----------------------------------------------------------------------------
# Step 1: Demographics statistics → Excel
# -----------------------------------------------------------------------------
print("\n[Step 1] Demographics statistics...")

excel_path = '/content/demographics_statistics.xlsx'
writer     = pd.ExcelWriter(excel_path, engine='openpyxl')

# Sheet 1: Overall summary by label
summary_data = []
for label in labels_sorted:
    subset = df[df['label'] == label]
    row    = {'Label': label, 'N': len(subset),
              'N_%': f"{len(subset)/len(df)*100:.1f}%"}

    if 'age' in df.columns:
        row.update({'Age_Mean': subset['age'].mean(), 'Age_SD': subset['age'].std(),
                    'Age_Min': subset['age'].min(), 'Age_Max': subset['age'].max()})
    if 'sex' in df.columns:
        row.update({'Female_N': (subset['sex']==0).sum(),
                    'Male_N':   (subset['sex']==1).sum(),
                    'Male_%':   (subset['sex']==1).sum() / len(subset) * 100})
    if 'education' in df.columns:
        row.update({'Education_Mean': subset['education'].mean(),
                    'Education_SD':   subset['education'].std(),
                    'Education_Min':  subset['education'].min(),
                    'Education_Max':  subset['education'].max()})
    if 'disease duration' in df.columns:
        row.update({'Disease_Duration_Mean': subset['disease duration'].mean(),
                    'Disease_Duration_SD':   subset['disease duration'].std(),
                    'Disease_Duration_Min':  subset['disease duration'].min(),
                    'Disease_Duration_Max':  subset['disease duration'].max()})
    summary_data.append(row)

# Total row
total_row = {'Label': 'Total', 'N': len(df), 'N_%': '100.0%'}
if 'age' in df.columns:
    total_row.update({'Age_Mean': df['age'].mean(), 'Age_SD': df['age'].std(),
                      'Age_Min': df['age'].min(), 'Age_Max': df['age'].max()})
if 'sex' in df.columns:
    total_row.update({'Female_N': (df['sex']==0).sum(), 'Male_N': (df['sex']==1).sum(),
                      'Male_%': (df['sex']==1).sum() / len(df) * 100})
if 'education' in df.columns:
    total_row.update({'Education_Mean': df['education'].mean(),
                      'Education_SD': df['education'].std(),
                      'Education_Min': df['education'].min(),
                      'Education_Max': df['education'].max()})
if 'disease duration' in df.columns:
    total_row.update({'Disease_Duration_Mean': df['disease duration'].mean(),
                      'Disease_Duration_SD': df['disease duration'].std(),
                      'Disease_Duration_Min': df['disease duration'].min(),
                      'Disease_Duration_Max': df['disease duration'].max()})
summary_data.append(total_row)

pd.DataFrame(summary_data).to_excel(writer, sheet_name='Overall_Summary', index=False)
print(f"  Overall Summary: {len(summary_data)} rows")

# Sheet 2: Statistical tests for demographics
test_results = []
if 'age' in df.columns:
    groups   = [df[df['label']==l]['age'].dropna() for l in labels_sorted]
    h, p     = kruskal(*groups)
    test_results.append({'Variable': 'Age', 'Test': 'Kruskal-Wallis',
                         'Statistic': h, 'p-value': p,
                         'Significant': 'Yes' if p < 0.05 else 'No',
                         'Significance': '***' if p < 0.001 else '**' if p < 0.01
                                         else '*' if p < 0.05 else 'ns'})
if 'sex' in df.columns:
    contingency = pd.crosstab(df['label'], df['sex'])
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    test_results.append({'Variable': 'Sex', 'Test': 'Chi-square',
                         'Statistic': chi2, 'p-value': p,
                         'Significant': 'Yes' if p < 0.05 else 'No',
                         'Significance': '***' if p < 0.001 else '**' if p < 0.01
                                         else '*' if p < 0.05 else 'ns'})
if 'education' in df.columns:
    groups = [df[df['label']==l]['education'].dropna() for l in labels_sorted]
    h, p   = kruskal(*groups)
    test_results.append({'Variable': 'Education', 'Test': 'Kruskal-Wallis',
                         'Statistic': h, 'p-value': p,
                         'Significant': 'Yes' if p < 0.05 else 'No',
                         'Significance': '***' if p < 0.001 else '**' if p < 0.01
                                         else '*' if p < 0.05 else 'ns'})
if 'disease duration' in df.columns:
    df_pd  = df[df['label'] != 'HC']
    groups = [df_pd[df_pd['label']==l]['disease duration'].dropna()
              for l in ['PD-NC', 'PD-MCI', 'PD-D']]
    h, p   = kruskal(*groups)
    test_results.append({'Variable': 'Disease Duration (PD only)', 'Test': 'Kruskal-Wallis',
                         'Statistic': h, 'p-value': p,
                         'Significant': 'Yes' if p < 0.05 else 'No',
                         'Significance': '***' if p < 0.001 else '**' if p < 0.01
                                         else '*' if p < 0.05 else 'ns'})

test_df = pd.DataFrame(test_results)
test_df.to_excel(writer, sheet_name='Statistical_Tests', index=False)
print(f"  Statistical Tests: {len(test_df)} tests")

# Sheet 3: Pairwise comparisons (age, education)
pairwise_results = []
for var in ['age', 'education']:
    if var not in df.columns:
        continue
    for l1, l2 in combinations(labels_sorted, 2):
        g1, g2 = df[df['label']==l1][var].dropna(), df[df['label']==l2][var].dropna()
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        mean_diff   = g1.mean() - g2.mean()
        pooled_std  = np.sqrt(((len(g1)-1)*g1.std()**2 + (len(g2)-1)*g2.std()**2) /
                              (len(g1) + len(g2) - 2))
        cohens_d    = mean_diff / pooled_std if pooled_std > 0 else 0
        pairwise_results.append({
            'Variable': var.capitalize(), 'Group_1': l1, 'Group_2': l2,
            'Mean_1': g1.mean(), 'Mean_2': g2.mean(), 'Mean_Diff': mean_diff,
            'U_statistic': stat, 'p-value': p, 'Cohens_d': cohens_d,
            'Significant': 'Yes' if p < 0.05 else 'No'
        })

pd.DataFrame(pairwise_results).to_excel(writer, sheet_name='Pairwise_Comparisons', index=False)
print(f"  Pairwise Comparisons: {len(pairwise_results)} tests")

# Sheet 4: By cohort
cohort_data = []
for cohort in df['Sources'].unique():
    subset = df[df['Sources'] == cohort]
    for label in labels_sorted:
        ls = subset[subset['label'] == label]
        if len(ls) == 0:
            continue
        row = {'Cohort': cohort, 'Label': label, 'N': len(ls)}
        if 'age' in df.columns:
            row.update({'Age_Mean': ls['age'].mean(), 'Age_SD': ls['age'].std()})
        if 'sex' in df.columns:
            row['Male_%'] = (ls['sex']==1).sum() / len(ls) * 100
        if 'education' in df.columns:
            row.update({'Education_Mean': ls['education'].mean(),
                        'Education_SD': ls['education'].std()})
        if 'disease duration' in df.columns:
            row.update({'Disease_Duration_Mean': ls['disease duration'].mean(),
                        'Disease_Duration_SD': ls['disease duration'].std()})
        cohort_data.append(row)

pd.DataFrame(cohort_data).to_excel(writer, sheet_name='By_Cohort', index=False)
writer.close()
print(f"  Saved: {excel_path}")

# -----------------------------------------------------------------------------
# Step 2: Biomarker validation → Excel + PDF
# -----------------------------------------------------------------------------
print("\n[Step 2] Biomarker validation...")

# Kruskal-Wallis for all biomarkers
kw_results = []
for biomarker in ALL_BIOMARKERS:
    if biomarker not in df.columns:
        continue
    groups = [df[df['label']==l][biomarker].dropna() for l in labels_sorted]
    h, p   = kruskal(*groups)
    family = ('THT' if biomarker in BIOMARKERS['THT'] else
              'DLS' if biomarker in BIOMARKERS['DLS'] else 'Neurotoxicity')
    kw_results.append({
        'Biomarker': biomarker, 'Family': family,
        'H_statistic': h, 'p-value': p,
        'Significant': 'Yes' if p < 0.05 else 'No',
        'Significance': '***' if p < 0.001 else '**' if p < 0.01
                        else '*' if p < 0.05 else 'ns'
    })

kw_df = pd.DataFrame(kw_results).sort_values('p-value')
print(f"  Significant biomarkers: {len(kw_df[kw_df['Significant']=='Yes'])} / {len(kw_df)}")

# Biomarker stats by group
biomarker_stats = []
for biomarker in ALL_BIOMARKERS:
    if biomarker not in df.columns:
        continue
    row = {'Biomarker': biomarker,
           'Overall_Mean': df[biomarker].mean(),
           'Overall_SD':   df[biomarker].std()}
    for label in labels_sorted:
        sub = df[df['label']==label][biomarker]
        row.update({f'{label}_Mean': sub.mean(), f'{label}_SD': sub.std(),
                    f'{label}_N': len(sub.dropna())})
    biomarker_stats.append(row)

# Pairwise comparisons for top 5 biomarkers
top5 = kw_df.nsmallest(5, 'p-value')['Biomarker'].tolist()
pairwise_bio = []
for biomarker in top5:
    for l1, l2 in combinations(labels_sorted, 2):
        g1, g2 = df[df['label']==l1][biomarker].dropna(), df[df['label']==l2][biomarker].dropna()
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        mean_diff  = g1.mean() - g2.mean()
        pooled_std = np.sqrt(((len(g1)-1)*g1.std()**2 + (len(g2)-1)*g2.std()**2) /
                             (len(g1) + len(g2) - 2))
        cohens_d   = mean_diff / pooled_std if pooled_std > 0 else 0
        pairwise_bio.append({
            'Biomarker': biomarker, 'Group_1': l1, 'Group_2': l2,
            'Mean_1': g1.mean(), 'Mean_2': g2.mean(), 'Mean_Diff': mean_diff,
            'U_statistic': stat, 'p-value': p, 'Cohens_d': cohens_d,
            'Effect_Size': 'Large' if abs(cohens_d)>0.8 else 'Medium' if abs(cohens_d)>0.5
                           else 'Small' if abs(cohens_d)>0.2 else 'Negligible',
            'Significant': 'Yes' if p < 0.05 else 'No'
        })

# Save biomarker Excel
bio_excel_path = '/content/biomarker_validation.xlsx'
with pd.ExcelWriter(bio_excel_path, engine='openpyxl') as bw:
    pd.DataFrame(biomarker_stats).to_excel(bw, sheet_name='Statistics_by_Group', index=False)
    kw_df.to_excel(bw, sheet_name='Kruskal_Wallis_Tests', index=False)
    pd.DataFrame(pairwise_bio).to_excel(bw, sheet_name='Pairwise_Comparisons', index=False)
print(f"  Saved: {bio_excel_path}")

# Biomarker figures PDF
pdf_path = '/content/biomarker_validation_figures.pdf'
with PdfPages(pdf_path) as pdf:

    # Figure 1: Significance bar chart
    fig, ax = plt.subplots(figsize=(12, 8))
    kw_plot = kw_df.copy()
    kw_plot['neg_log_p'] = -np.log10(kw_plot['p-value'])
    kw_plot = kw_plot.sort_values('Family')
    colors  = ['#1f77b4' if f=='THT' else '#ff7f0e' if f=='DLS' else '#2ca02c'
               for f in kw_plot['Family']]
    ax.barh(kw_plot['Biomarker'], kw_plot['neg_log_p'], color=colors)
    ax.axvline(-np.log10(0.05),  color='red',     linestyle='--', lw=2, label='p=0.05')
    ax.axvline(-np.log10(0.001), color='darkred', linestyle='--', lw=2, label='p=0.001')
    ax.set_xlabel('-log10(p-value)', fontsize=12)
    ax.set_title('Biomarker Discriminative Power (4-group Kruskal-Wallis)',
                 fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    pdf.savefig(fig, dpi=300)
    plt.close()

    # Figure 2: Top 5 boxplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    for idx, biomarker in enumerate(top5):
        ax = axes[idx]
        plot_data   = [df[df['label']==l][biomarker].dropna() for l in labels_sorted]
        bp = ax.boxplot(plot_data, labels=labels_sorted, patch_artist=True)
        for patch, color in zip(bp['boxes'],
                                ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        p_val = kw_df[kw_df['Biomarker']==biomarker]['p-value'].values[0]
        sig   = kw_df[kw_df['Biomarker']==biomarker]['Significance'].values[0]
        ax.set_title(f'{biomarker}\np={p_val:.4f} {sig}', fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
    axes[5].axis('off')
    plt.suptitle('Top 5 Most Discriminative Biomarkers', fontsize=16, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, dpi=300)
    plt.close()

    # Figure 3: Correlation heatmap
    all_feats   = ALL_BIOMARKERS + ['age', 'sex', 'education', 'disease duration']
    feats_avail = [f for f in all_feats if f in df.columns]
    corr_matrix = df[feats_avail].corr()
    fig, ax = plt.subplots(figsize=(16, 14))
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
                vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    pdf.savefig(fig, dpi=300)
    plt.close()

print(f"  Saved: {pdf_path}")

# =============================================================================
# PART 2: ML CLASSIFICATION ANALYSIS
#         Phase 1 (Univariate) / Phase 2 (Top-K) / Phase 3 (Predefined sets)
# =============================================================================

print("\n" + "=" * 70)
print("PART 2: ML CLASSIFICATION ANALYSIS")
print("=" * 70)

# -----------------------------------------------------------------------------
# Model definitions
# -----------------------------------------------------------------------------
def get_models(exclude_catboost=False):
    models = {
        'GBDT': GradientBoostingClassifier(
            random_state=RANDOM_STATE, n_estimators=100, subsample=0.8),
        'XGBoost': XGBClassifier(
            random_state=RANDOM_STATE, use_label_encoder=False,
            eval_metric='logloss', verbosity=0),
        'Random Forest': RandomForestClassifier(
            random_state=RANDOM_STATE, n_estimators=100, class_weight='balanced'),
        'Decision Tree': DecisionTreeClassifier(
            random_state=RANDOM_STATE, class_weight='balanced'),
        'Extra Trees': ExtraTreesClassifier(
            random_state=RANDOM_STATE, n_estimators=100, class_weight='balanced'),
        'Logistic Regression': LogisticRegression(
            penalty='l1', solver='saga', max_iter=2000,
            random_state=RANDOM_STATE, class_weight='balanced'),
    }
    if not exclude_catboost:
        models['CatBoost'] = CatBoostClassifier(
            random_state=RANDOM_STATE, verbose=0, iterations=100,
            depth=6, learning_rate=0.1, loss_function='Logloss')
    return models

def clone_model(model):
    if isinstance(model, LogisticRegression):
        return LogisticRegression(penalty='l1', solver='saga', max_iter=2000,
                                  random_state=RANDOM_STATE, class_weight='balanced')
    elif isinstance(model, CatBoostClassifier):
        return CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, iterations=100,
                                  depth=6, learning_rate=0.1, loss_function='Logloss')
    elif isinstance(model, XGBClassifier):
        return XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False,
                             eval_metric='logloss', verbosity=0)
    else:
        return deepcopy(model)

def calc_metrics(y_true, y_pred, y_proba, is_multiclass=False):
    try:
        if is_multiclass:
            auc  = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
            prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
            rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
            f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
        else:
            auc  = roc_auc_score(y_true, y_proba)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec  = recall_score(y_true, y_pred, zero_division=0)
            f1   = f1_score(y_true, y_pred, zero_division=0)
        return {'AUC': auc, 'Accuracy': accuracy_score(y_true, y_pred),
                'Precision': prec, 'Recall': rec, 'F1': f1}
    except:
        return {'AUC': 0, 'Accuracy': 0, 'Precision': 0, 'Recall': 0, 'F1': 0}

# -----------------------------------------------------------------------------
# Task definitions
# -----------------------------------------------------------------------------
label_map = {'HC': 0, 'PD-NC': 1, 'PD-MCI': 2, 'PD-D': 3}
df_pd     = df[df['label'] != 'HC'].copy()

TASKS = {
    'Task1_HC_vs_PD': {
        'X': df[FEATURES_TASK1].copy(),
        'y': (df['label'] != 'HC').astype(int),
        'name': 'HC vs PD', 'n_classes': 2,
        'class_names': ['HC', 'PD'],
        'features': FEATURES_TASK1, 'is_4class': False
    },
    'Task2_NC_vs_CI': {
        'X': df_pd[FEATURES_TASK23].copy(),
        'y': (df_pd['label'] != 'PD-NC').astype(int),
        'name': 'PD-NC vs PD-CI', 'n_classes': 2,
        'class_names': ['PD-NC', 'PD-CI'],
        'features': FEATURES_TASK23, 'is_4class': False
    },
    'Task3_4CLASS': {
        'X': df[FEATURES_TASK23].copy(),
        'y': df['label'].map(label_map),
        'name': '4-class', 'n_classes': 4,
        'class_names': ['HC', 'PD-NC', 'PD-MCI', 'PD-D'],
        'features': FEATURES_TASK23, 'is_4class': True
    }
}

for tid, t in TASKS.items():
    print(f"  {tid}: {len(t['y'])} samples, {len(t['features'])} features")

# -----------------------------------------------------------------------------
# Phase 1: Univariate
# -----------------------------------------------------------------------------
def run_phase1(task_id, task_config):
    print(f"\n[Phase 1] {task_config['name']}")

    X, y          = task_config['X'], task_config['y']
    features      = task_config['features']
    is_multiclass = task_config['n_classes'] > 2
    is_4class     = task_config['is_4class']
    models        = get_models(is_4class)
    cv            = RepeatedStratifiedKFold(n_splits=5, n_repeats=10,
                                            random_state=RANDOM_STATE)
    results = []

    for idx, feature in enumerate(features, 1):
        X_feat = X[[feature]]
        print(f"  [{idx}/{len(features)}] {feature:<35}", end=" ")

        for model_name, model in models.items():
            try:
                fold_metrics = []
                for train_idx, test_idx in cv.split(X_feat, y):
                    X_tr, X_te = X_feat.iloc[train_idx], X_feat.iloc[test_idx]
                    y_tr, y_te = y.iloc[train_idx].values, y.iloc[test_idx].values
                    m = clone_model(model)
                    m.fit(X_tr, y_tr)
                    y_pred  = m.predict(X_te)
                    y_proba = m.predict_proba(X_te) if hasattr(m, 'predict_proba') \
                              else m.decision_function(X_te)
                    if not is_multiclass and y_proba.ndim == 2:
                        y_proba = y_proba[:, 1]
                    fold_metrics.append(calc_metrics(y_te, y_pred, y_proba, is_multiclass))

                row = {'Task': task_config['name'], 'Feature': feature, 'Model': model_name}
                for metric in ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']:
                    vals = [m[metric] for m in fold_metrics]
                    row[f'{metric}_mean'] = np.mean(vals)
                    row[f'{metric}_std']  = np.std(vals)
                results.append(row)
            except Exception:
                continue
        print("done")

    df_res = pd.DataFrame(results)
    out_path = OUTPUT_PATH / f"{task_id}_Phase1_Results.xlsx"
    df_res.to_excel(out_path, index=False)
    print(f"  Saved: {out_path.name}")

# -----------------------------------------------------------------------------
# Phase 2: Top-K
# -----------------------------------------------------------------------------
def run_phase2(task_id, task_config):
    print(f"\n[Phase 2] {task_config['name']}")

    X, y          = task_config['X'], task_config['y']
    features      = task_config['features']
    class_names   = task_config['class_names']
    is_multiclass = task_config['n_classes'] > 2
    is_4class     = task_config['is_4class']

    # RF feature importance
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                class_weight='balanced')
    rf.fit(X, y)
    importances = rf.feature_importances_
    sorted_idx  = np.argsort(importances)[::-1]

    # Feature importance table
    fi_rows = [{'Task': task_config['name'], 'Rank': r+1,
                'Feature': features[i], 'Importance': importances[i]}
               for r, i in enumerate(sorted_idx)]
    df_fi = pd.DataFrame(fi_rows)
    df_fi.to_excel(OUTPUT_PATH / f"{task_id}_Phase2_FeatureImportance.xlsx", index=False)

    print("  Top 10 features (RF importance):")
    for r in range(min(10, len(features))):
        print(f"    {r+1}. {features[sorted_idx[r]]:<35} {importances[sorted_idx[r]]:.4f}")

    # Top-K values
    n = len(features)
    top_k_values = [1, 2, 3, 5, 7, 10, 13, 15, n]

    models  = get_models(is_4class)
    cv      = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    results = []
    cms, reports = {}, {}

    for top_k in top_k_values:
        selected = [features[i] for i in sorted_idx[:top_k]]
        X_sub    = X[selected]
        print(f"\n  Top-{top_k}: {', '.join(selected[:3])}{'...' if top_k > 3 else ''}")

        for model_name, model in models.items():
            try:
                fold_metrics  = []
                y_pred_all, y_true_all = [], []

                for train_idx, test_idx in cv.split(X_sub, y):
                    X_tr, X_te = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
                    y_tr, y_te = y.iloc[train_idx].values, y.iloc[test_idx].values
                    m = clone_model(model)
                    m.fit(X_tr, y_tr)
                    y_pred  = m.predict(X_te)
                    y_proba = m.predict_proba(X_te) if hasattr(m, 'predict_proba') \
                              else m.decision_function(X_te)
                    if not is_multiclass and y_proba.ndim == 2:
                        y_proba = y_proba[:, 1]
                    y_pred_all.extend(y_pred)
                    y_true_all.extend(y_te)
                    fold_metrics.append(calc_metrics(y_te, y_pred, y_proba, is_multiclass))

                row = {'Task': task_config['name'], 'Top_K': top_k,
                       'Model': model_name, 'Features': ', '.join(selected)}
                for metric in ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']:
                    vals = [m[metric] for m in fold_metrics]
                    row[f'{metric}_mean'] = np.mean(vals)
                    row[f'{metric}_std']  = np.std(vals)
                results.append(row)

                key = f"TopK{top_k}_{model_name}"
                cms[key]     = confusion_matrix(y_true_all, y_pred_all)
                reports[key] = classification_report(y_true_all, y_pred_all,
                                                     target_names=class_names,
                                                     output_dict=True, zero_division=0)
            except Exception:
                continue
        print("    done")

    # Save results
    pd.DataFrame(results).to_excel(
        OUTPUT_PATH / f"{task_id}_Phase2_Results.xlsx", index=False)

    with pd.ExcelWriter(OUTPUT_PATH / f"{task_id}_Phase2_ConfusionMatrices.xlsx",
                        engine='openpyxl') as w:
        for key, cm in cms.items():
            pd.DataFrame(cm, index=class_names,
                         columns=class_names).to_excel(w, sheet_name=key[:31])

    reports_rows = [{'Config': k, 'Class': cls, **m}
                    for k, rep in reports.items()
                    for cls, m in rep.items() if isinstance(m, dict)]
    pd.DataFrame(reports_rows).to_excel(
        OUTPUT_PATH / f"{task_id}_Phase2_ClassificationReports.xlsx", index=False)

    print(f"  Saved: {task_id}_Phase2_*.xlsx")

# -----------------------------------------------------------------------------
# Phase 3: Predefined feature sets
# -----------------------------------------------------------------------------
def run_phase3(task_id, task_config):
    print(f"\n[Phase 3] {task_config['name']}")

    X, y          = task_config['X'], task_config['y']
    features      = task_config['features']
    class_names   = task_config['class_names']
    is_multiclass = task_config['n_classes'] > 2
    is_4class     = task_config['is_4class']
    has_duration  = 'disease duration' in features

    feature_sets = {
        'tht':                BIOMARKERS['THT'],
        'dls':                BIOMARKERS['DLS'],
        'neurotoxicity':      BIOMARKERS['Neurotoxicity'],
        'all_biomarkers':     ALL_BIOMARKERS,
        'sex_only':           ['sex'],
        'age_only':           ['age'],
        'education_only':     ['education'],
        'biomarkers_sex':     ALL_BIOMARKERS + ['sex'],
        'biomarkers_age':     ALL_BIOMARKERS + ['age'],
        'biomarkers_education': ALL_BIOMARKERS + ['education'],
        'all_features':       features
    }
    if has_duration:
        feature_sets['duration_only']       = ['disease duration']
        feature_sets['biomarkers_duration'] = ALL_BIOMARKERS + ['disease duration']

    models  = get_models(is_4class)
    cv      = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    results = []
    cms, reports = {}, {}

    for set_name, feat_list in feature_sets.items():
        X_sub = X[feat_list]
        print(f"\n  {set_name} ({len(feat_list)} features)")

        for model_name, model in models.items():
            try:
                fold_metrics  = []
                y_pred_all, y_true_all = [], []

                for train_idx, test_idx in cv.split(X_sub, y):
                    X_tr, X_te = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
                    y_tr, y_te = y.iloc[train_idx].values, y.iloc[test_idx].values
                    m = clone_model(model)
                    m.fit(X_tr, y_tr)
                    y_pred  = m.predict(X_te)
                    y_proba = m.predict_proba(X_te) if hasattr(m, 'predict_proba') \
                              else m.decision_function(X_te)
                    if not is_multiclass and y_proba.ndim == 2:
                        y_proba = y_proba[:, 1]
                    y_pred_all.extend(y_pred)
                    y_true_all.extend(y_te)
                    fold_metrics.append(calc_metrics(y_te, y_pred, y_proba, is_multiclass))

                row = {'Task': task_config['name'], 'FeatureSet': set_name,
                       'N_Features': len(feat_list), 'Model': model_name}
                for metric in ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']:
                    vals = [m[metric] for m in fold_metrics]
                    row[f'{metric}_mean'] = np.mean(vals)
                    row[f'{metric}_std']  = np.std(vals)
                results.append(row)

                key = f"{set_name}_{model_name}"
                cms[key]     = confusion_matrix(y_true_all, y_pred_all)
                reports[key] = classification_report(y_true_all, y_pred_all,
                                                     target_names=class_names,
                                                     output_dict=True, zero_division=0)
            except Exception:
                continue
        print("    done")

    # Save results
    pd.DataFrame(results).to_excel(
        OUTPUT_PATH / f"{task_id}_Phase3_Results.xlsx", index=False)

    with pd.ExcelWriter(OUTPUT_PATH / f"{task_id}_Phase3_ConfusionMatrices.xlsx",
                        engine='openpyxl') as w:
        for key, cm in cms.items():
            pd.DataFrame(cm, index=class_names,
                         columns=class_names).to_excel(w, sheet_name=key[:31])

    reports_rows = [{'Config': k, 'Class': cls, **m}
                    for k, rep in reports.items()
                    for cls, m in rep.items() if isinstance(m, dict)]
    pd.DataFrame(reports_rows).to_excel(
        OUTPUT_PATH / f"{task_id}_Phase3_ClassificationReports.xlsx", index=False)

    print(f"  Saved: {task_id}_Phase3_*.xlsx")

# -----------------------------------------------------------------------------
# Run all tasks
# -----------------------------------------------------------------------------
print("\n[Main] Running Phase 1, 2, 3 for all tasks...")

for task_id, task_config in TASKS.items():
    print(f"\n{'#'*70}")
    print(f"# {task_id}: {task_config['name']}")
    print(f"{'#'*70}")
    run_phase1(task_id, task_config)
    run_phase2(task_id, task_config)
    run_phase3(task_id, task_config)

# =============================================================================
# PART 3: LEAVE-ONE-COHORT-OUT VALIDATION
#         Best models: XGBoost (Task1), Extra Trees (Task2, 3)
# =============================================================================

print("\n" + "=" * 70)
print("PART 3: LEAVE-ONE-COHORT-OUT VALIDATION")
print("=" * 70)

print(f"\nJHU: {(df['Sources']=='JHU').sum()} samples")
print(f"UW:  {(df['Sources']=='UW').sum()} samples")

# Load feature importance (top-K from Phase 2)
df_fi1 = pd.read_excel(OUTPUT_PATH / "Task1_HC_vs_PD_Phase2_FeatureImportance.xlsx")
df_fi2 = pd.read_excel(OUTPUT_PATH / "Task2_NC_vs_CI_Phase2_FeatureImportance.xlsx")
df_fi3 = pd.read_excel(OUTPUT_PATH / "Task3_4CLASS_Phase2_FeatureImportance.xlsx")

sorted_feats1 = df_fi1.sort_values('Rank')['Feature'].tolist()[:13]
sorted_feats2 = df_fi2.sort_values('Rank')['Feature'].tolist()[:17]
sorted_feats3 = df_fi3.sort_values('Rank')['Feature'].tolist()[:13]

loco_configs = {
    'Task1_HC_vs_PD': {
        'model':       XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False,
                                     eval_metric='logloss', verbosity=0),
        'features':    sorted_feats1,
        'name':        'HC vs PD',
        'get_y':       lambda d: (d['label'] != 'HC').astype(int),
        'class_names': ['HC', 'PD']
    },
    'Task2_NC_vs_CI': {
        'model':       ExtraTreesClassifier(random_state=RANDOM_STATE, n_estimators=100,
                                            class_weight='balanced'),
        'features':    sorted_feats2,
        'name':        'PD-NC vs PD-CI',
        'get_y':       lambda d: (d['label'] != 'PD-NC').astype(int),
        'filter':      lambda d: d[d['label'] != 'HC'],
        'class_names': ['PD-NC', 'PD-CI']
    },
    'Task3_4CLASS': {
        'model':       ExtraTreesClassifier(random_state=RANDOM_STATE, n_estimators=100,
                                            class_weight='balanced'),
        'features':    sorted_feats3,
        'name':        '4-class',
        'get_y':       lambda d: d['label'].map({'HC':0,'PD-NC':1,'PD-MCI':2,'PD-D':3}),
        'class_names': ['HC', 'PD-NC', 'PD-MCI', 'PD-D']
    }
}

loco_results = []

for task_id, config in loco_configs.items():
    print(f"\n[{task_id}]")

    df_task   = config['filter'](df) if 'filter' in config else df
    df_jhu    = df_task[df_task['Sources'] == 'JHU']
    df_uw     = df_task[df_task['Sources'] == 'UW']
    feats     = config['features']
    is_mc     = len(config['class_names']) > 2

    X_jhu, y_jhu = df_jhu[feats], config['get_y'](df_jhu)
    X_uw,  y_uw  = df_uw[feats],  config['get_y'](df_uw)

    print(f"  JHU: {len(X_jhu)}, UW: {len(X_uw)}")

    for scenario, (X_tr, y_tr, X_te, y_te) in {
        'Train JHU → Test UW': (X_jhu, y_jhu, X_uw, y_uw),
        'Train UW → Test JHU': (X_uw,  y_uw,  X_jhu, y_jhu)
    }.items():
        m = deepcopy(config['model'])
        m.fit(X_tr, y_tr)
        y_pred  = m.predict(X_te)
        y_proba = m.predict_proba(X_te) if hasattr(m, 'predict_proba') \
                  else m.decision_function(X_te)
        if not is_mc and y_proba.ndim == 2:
            y_proba = y_proba[:, 1]

        if is_mc:
            auc  = roc_auc_score(y_te, y_proba, multi_class='ovr', average='macro')
            prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
            rec  = recall_score(y_te, y_pred, average='macro', zero_division=0)
            f1   = f1_score(y_te, y_pred, average='macro', zero_division=0)
        else:
            auc  = roc_auc_score(y_te, y_proba)
            prec = precision_score(y_te, y_pred, zero_division=0)
            rec  = recall_score(y_te, y_pred, zero_division=0)
            f1   = f1_score(y_te, y_pred, zero_division=0)
        acc = accuracy_score(y_te, y_pred)

        print(f"  {scenario}: AUC={auc:.3f}, Acc={acc:.3f}, F1={f1:.3f}")
        loco_results.append({
            'Task': config['name'], 'Scenario': scenario,
            'Train_N': len(X_tr), 'Test_N': len(X_te),
            'AUC': auc, 'Accuracy': acc,
            'Precision': prec, 'Recall': rec, 'F1': f1
        })

    # Add CV results from Phase 2
    df_p2 = pd.read_excel(OUTPUT_PATH / f"{task_id}_Phase2_Results.xlsx")
    if task_id == 'Task1_HC_vs_PD':
        row = df_p2[(df_p2['Top_K']==13) & (df_p2['Model']=='XGBoost')].iloc[0]
    elif task_id == 'Task2_NC_vs_CI':
        row = df_p2[(df_p2['Top_K']==17) & (df_p2['Model']=='Extra Trees')].iloc[0]
    else:
        row = df_p2[(df_p2['Top_K']==13) & (df_p2['Model']=='Extra Trees')].iloc[0]

    loco_results.append({
        'Task': config['name'], 'Scenario': 'Combined (5x10 CV)',
        'Train_N': len(df_task), 'Test_N': len(df_task),
        'AUC': row['AUC_mean'], 'Accuracy': row['Accuracy_mean'],
        'Precision': row['Precision_mean'], 'Recall': row['Recall_mean'],
        'F1': row['F1_mean']
    })

df_loco = pd.DataFrame(loco_results)
loco_path = OUTPUT_PATH / 'LeaveOneCohortOut_Results.xlsx'
df_loco.to_excel(loco_path, index=False)
print(f"\nSaved: {loco_path}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

print(f"""
OUTPUT FILES:
  /content/
    demographics_statistics.xlsx
    biomarker_validation.xlsx
    biomarker_validation_figures.pdf

  /content/final_corrected_results/
    Task1_HC_vs_PD_Phase1_Results.xlsx
    Task1_HC_vs_PD_Phase2_Results.xlsx
    Task1_HC_vs_PD_Phase2_FeatureImportance.xlsx
    Task1_HC_vs_PD_Phase2_ConfusionMatrices.xlsx
    Task1_HC_vs_PD_Phase2_ClassificationReports.xlsx
    Task1_HC_vs_PD_Phase3_Results.xlsx
    Task1_HC_vs_PD_Phase3_ConfusionMatrices.xlsx
    Task1_HC_vs_PD_Phase3_ClassificationReports.xlsx
    (Task2, Task3 동일 구조)
    LeaveOneCohortOut_Results.xlsx

CV:     RepeatedStratifiedKFold (5-fold x 10 repeats)
Models: GBDT, XGBoost, Random Forest, Decision Tree,
        Extra Trees, Logistic Regression, CatBoost
""")